$$
\text{DC: Direct Current，直流, 也可以是DC voltage电压或DC current电流的方向基本不变} \\
V(t) = 5V \\ \quad \\
\text{AC: Alternating Current，交流, 信号随时间变化} \\
V(t) = A \sin(2 \pi f t) \\ \quad \\
\text{V(t): 瞬时值 / Instantaneous Value} \\
\text{A: 振幅 / Amplitude} \\
\text{f: 频率 / Frequency} \\
\text{t: 时间 / Time} \\ \quad \\

2 \pi f \text{ 或 } \omega \text{: 角频率 / Angular Frequency} \\
2 \pi f t\text{: 相位 / Phase (更一般地 } V(t) = A \sin(2 \pi f t + \varphi) \text{, } \varphi \text{ 是初相位 / Initial Phase)} \\
\text{T: } \frac{1}{f} \text{周期 / Period} \\
\quad 
$$

In [ ]:
import numpy as np

# 写一个 V(t) = A \sin(2 \pi f t)

A = 10  # 振幅
f = 50  # 频率，单位：赫兹

AC_signal = lambda t: A * np.sin(2 * np.pi * f * t)


<a id="rms"></a>

$$
\text{有效值 (Root Mean Square, RMS): 和交流电在同一个电阻上产生相同平均功率的直流电压} \\ \quad \\
\text{推导: 电阻上的瞬时功率 } p(t) = \frac{v(t)^2}{R} \text{, 一个周期内的平均功率} \\
\bar{P} = \frac{1}{T} \int_0^T \frac{v(t)^2}{R} dt \\
\text{令它等于直流电压 } V_{rms} \text{ 的功率 } \frac{V_{rms}^2}{R} \text{, R 约掉:} \\
V_{rms} = \sqrt{\frac{1}{T} \int_0^T v(t)^2 \, dt} \quad (\text{平方 Square, 平均 Mean, 开方 Root}) \\ \quad \\
\text{正弦波 } v(t) = A \sin(\omega t) \text{, } \omega = \frac{2\pi}{T} \text{:} \\
\text{由 } \cos 2\theta = 1 - 2 \sin^2\theta \implies \sin^2\theta = \frac{1 - \cos 2\theta}{2} \\
\int_0^T A^2 \sin^2(\omega t) \, dt = \frac{A^2}{2} \int_0^T dt - \frac{A^2}{2} \underbrace{\int_0^T \cos(2\omega t) \, dt}_{= \frac{\sin(2\omega T)}{2\omega} = \frac{\sin 4\pi}{2\omega} = 0} = \frac{A^2 T}{2} \\
V_{rms} = \sqrt{\frac{1}{T} \cdot \frac{A^2 T}{2}} = \frac{A}{\sqrt{2}} \approx 0.707 A \\ \quad \\
\text{例: 家用电 220 V 指的是有效值, 峰值 } A = 220 \sqrt{2} \approx 311 \text{ V} \\ \quad \\
\text{其他波形 (同样代入定义积分):} \\
\text{方波 (Square Wave, } \pm A \text{): } v^2 = A^2 \text{ 恒定} \implies V_{rms} = A \\
\text{三角波 (Triangle Wave): 在 } [0, \tfrac{T}{4}] \text{ 上 } v = \frac{4A}{T} t \text{, 由对称性只需算四分之一周期} \\
\frac{4}{T} \int_0^{T/4} \frac{16 A^2}{T^2} t^2 dt = \frac{64 A^2}{T^3} \cdot \frac{1}{3} \left(\frac{T}{4}\right)^3 = \frac{A^2}{3} \implies V_{rms} = \frac{A}{\sqrt{3}}
$$

数学基础: [定积分与牛顿-莱布尼茨公式](../math_basics/calculus.ipynb#integral) · [数值积分 (梯形法)](../math_basics/calculus.ipynb#numerical-integral)

In [ ]:
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid, quad
from scipy import signal

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

T = 1 / f
t = np.linspace(0, T, 10001)

def rms_trapezoid(v, t):
    # 定义: sqrt( (1/T) * 积分 v^2 dt ), 积分用梯形法
    return np.sqrt(trapezoid(v**2, t) / (t[-1] - t[0]))

sine = AC_signal(t)
square = A * signal.square(2 * np.pi * f * t)
triangle = A * signal.sawtooth(2 * np.pi * f * t + np.pi / 2, width=0.5)

print(f'正弦波: 梯形法 RMS = {rms_trapezoid(sine, t):.6f}, 推导 A/sqrt(2) = {A / np.sqrt(2):.6f}, '
      f'quad = {np.sqrt(quad(lambda s: AC_signal(s)**2, 0, T)[0] / T):.6f}')
print(f'方波:   梯形法 RMS = {rms_trapezoid(square, t):.6f}, 推导 A = {A}')
print(f'三角波: 梯形法 RMS = {rms_trapezoid(triangle, t):.6f}, 推导 A/sqrt(3) = {A / np.sqrt(3):.6f}')
print(f'家用电: 有效值 220 V -> 峰值 = {220 * np.sqrt(2):.2f} V')

plt.figure(figsize=(4, 3))
plt.plot(t * 1000, sine, label='sine')
plt.plot(t * 1000, square, label='square')
plt.plot(t * 1000, triangle, label='triangle')
plt.axhline(A / np.sqrt(2), color='C0', linestyle='--', label='sine RMS')
plt.xlabel('t (ms)')
plt.ylabel('V')
plot_nicely()
plt.show()

<a id="dc-offset"></a>

$$
\text{直流 + 交流 (DC Offset): } v(t) = V_{dc} + A \sin(\omega t) \\
v^2 = V_{dc}^2 + 2 V_{dc} A \sin(\omega t) + A^2 \sin^2(\omega t) \\
\text{一个周期内平均: 中间的交叉项 } \sin(\omega t) \text{ 的平均为 0} \\
\overline{v^2} = V_{dc}^2 + 0 + \frac{A^2}{2} \implies V_{rms} = \sqrt{V_{dc}^2 + \left(\frac{A}{\sqrt{2}}\right)^2} \\
\text{直流分量和交流分量的 "功率" 直接相加 (正交 orthogonal, 傅里叶级数的帕塞瓦尔定理 Parseval 的特例)}
$$

In [ ]:
V_dc = 3.0
v_mix = V_dc + AC_signal(t)
print(f'DC + AC: 梯形法 RMS = {rms_trapezoid(v_mix, t):.6f}, 推导 sqrt(V_dc^2 + A^2/2) = {np.sqrt(V_dc**2 + A**2 / 2):.6f}')
print(f'交叉项的平均 = {trapezoid(2 * V_dc * AC_signal(t), t) / T:.2e} (== 0)')

<a id="ac-power"></a>

$$
\text{交流电的平均功率 (Average Power) 与功率因数 (Power Factor)} \\ \quad \\
\text{电压 } v(t) = V_m \sin(\omega t) \text{, 电流滞后相位差 } \varphi \text{: } i(t) = I_m \sin(\omega t - \varphi) \\
\text{(电阻 } \varphi = 0 \text{, 纯电感 } \varphi = 90^\circ \text{, 纯电容 } \varphi = -90^\circ \text{, 见 RLC 和相量那两个 note)} \\ \quad \\
\text{积化和差 (Product-to-Sum): 由 } \cos(a - b) = \cos a \cos b + \sin a \sin b, \ \cos(a + b) = \cos a \cos b - \sin a \sin b \\
\text{两式相减: } \cos(a - b) - \cos(a + b) = 2 \sin a \sin b \implies \sin a \sin b = \frac{1}{2}[\cos(a - b) - \cos(a + b)] \\ \quad \\
\text{瞬时功率: } p(t) = v(t) i(t) = V_m I_m \sin(\omega t) \sin(\omega t - \varphi) = \frac{V_m I_m}{2} \left[\underbrace{\cos\varphi}_{\text{常数}} - \underbrace{\cos(2\omega t - \varphi)}_{\text{二倍频振荡, 周期平均为 0}}\right] \\
\bar{P} = \frac{1}{T} \int_0^T p(t) \, dt = \frac{V_m I_m}{2} \cos\varphi = \frac{V_m}{\sqrt{2}} \cdot \frac{I_m}{\sqrt{2}} \cos\varphi = V_{rms} I_{rms} \cos\varphi \\ \quad \\
\text{有功功率 (Active Power) } P = V_{rms} I_{rms} \cos\varphi \text{ (W): 真正被消耗的功率} \\
\text{视在功率 (Apparent Power) } S = V_{rms} I_{rms} \text{ (VA)} \\
\text{无功功率 (Reactive Power) } Q = V_{rms} I_{rms} \sin\varphi \text{ (var): 在电源和电感/电容之间来回交换, 平均不消耗} \\
P^2 + Q^2 = S^2, \quad \text{功率因数 } PF = \frac{P}{S} = \cos\varphi \\
\text{纯电感 } \varphi = 90^\circ \implies P = 0 \text{: 有电流却不耗能 (但电流会让导线发热, 所以电网希望 PF 接近 1)}
$$

数学基础: [泰勒展开与欧拉公式 (三角恒等式的来源)](../math_basics/calculus.ipynb#taylor) · [定积分](../math_basics/calculus.ipynb#integral)

In [ ]:
V_m, I_m = 311.0, 5.0
for phi_deg in [0, 30, 60, 90, -45]:
    phi = np.deg2rad(phi_deg)
    v = V_m * np.sin(2 * np.pi * f * t)
    i = I_m * np.sin(2 * np.pi * f * t - phi)
    p = v * i
    P_numeric = trapezoid(p, t) / T
    P_formula = rms_trapezoid(v, t) * rms_trapezoid(i, t) * np.cos(phi)
    # 反过来: 由测得的平均功率和有效值求相位差
    phi_measured = np.rad2deg(np.arccos(P_numeric / (rms_trapezoid(v, t) * rms_trapezoid(i, t))))
    print(f'phi = {phi_deg:>4} 度: 数值平均功率 = {P_numeric:9.3f} W, V_rms I_rms cos(phi) = {P_formula:9.3f} W, '
          f'PF = {np.cos(phi):.4f}, 由功率反推 |phi| = {phi_measured:.2f} 度')

# 积化和差的数值验证
a_chk, b_chk = 0.7, 2.1
print(f'sin a sin b = {np.sin(a_chk) * np.sin(b_chk):.6f}, (cos(a-b) - cos(a+b))/2 = {(np.cos(a_chk - b_chk) - np.cos(a_chk + b_chk)) / 2:.6f}')

phi = np.deg2rad(60)
v = V_m * np.sin(2 * np.pi * f * t)
i = I_m * np.sin(2 * np.pi * f * t - phi)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(t * 1000, v, label='v(t)')
axs[0].plot(t * 1000, i * 40, label='i(t) x 40')
axs[0].set_xlabel('t (ms)')
axs[0].set_title('phase difference 60 deg')
axs[1].plot(t * 1000, v * i, label='p(t) = v i')
axs[1].axhline(trapezoid(v * i, t) / T, color='r', linestyle='--', label='average P')
axs[1].set_xlabel('t (ms)')
plot_nicely()
plt.show()